# coop_report

> Phase 12 report: Track A (instruction and channel variants for Claude, paired with Phase 11; metrics from the turn logs) and Track B (trained models against the warm-start model, Haiku and rule agents); `report.md` with provenance first, and figures. All findings are exploratory.

In [ ]:
#| default_exp coop_report

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import re

import matplotlib

matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

from explRL.bench_grid import rowKey
from explRL.llm_figures import _save
from explRL.llm_report import NAN, _f, _fmt, _paired, _sel, _stat

SUGGEST = re.compile(r'The coordinator suggests: T(\d+)\.')
KNOWN = re.compile(r'Your open subtask: T\d+, open\. Find x with 0 <= x < \d+\. Clues you know: (.*)\.$', re.M)
MECH = ('welfare', 'sharePerHeld', 'redundantShares')


def logMetrics(records, m):
    """Per row key, from turn logs (individual reward): follow share (opens of the coordinator's suggested subtask among
    opens with a suggestion) and solo solves (scoring answers submitted with fewer than m known clues)."""
    by = {}
    for x in records:
        if x.get('kind', 'turn') != 'turn':
            continue
        d, a = by.setdefault(x['key'], dict(follow=[], solo=0)), x['action']
        if a[0] == 'open' and (s := SUGGEST.search(x['obs'])):
            d['follow'].append(int(s.group(1)) == a[1])
        if a[0] == 'submit' and x['reward'] > 0 and (k := KNOWN.search(x['obs'])) and \
                len(re.findall(r'x mod', k.group(1))) < m:
            d['solo'] += 1
    return {k: dict(followShare=float(np.mean(d['follow'])) if d['follow'] else NAN, soloSolves=d['solo'])
            for k, d in by.items()}


def trackASummaries(rows, p11, logm=None, p11logm=None, study='coopA'):
    "TRACKA_PLAN: variant effects and coordinator gaps (paired on seed with Phase 11) and the mechanism metrics."
    a, base = _sel(rows, study=study, runner='llm'), _sel(p11, study='clue', runner='llm')
    S = dict(effect={}, gap={}, mechanism={})
    for v, ch, N in sorted({(r['level'], r['channel'], int(_f(r['N']))) for r in a}):
        cell, ref = _sel(a, level=v, channel=ch, N=N), _sel(base, channel=ch, N=N)
        S['effect'][(v, ch, N)] = _stat(_paired(cell, ref, 'welfare'))
        if ch == 'hierarchical':
            S['gap'][(v, ch, N)] = _stat(_paired(cell, _sel(base, channel='blackboard', N=N), 'welfare'))
        for name, rs, lm in (('variant', cell, logm), ('Phase 11', ref, p11logm)):
            mech = {m: _stat(_f(r[m]) for r in rs) for m in MECH}
            if lm is not None:
                mech.update({m: _stat(lm.get(rowKey(r), {}).get(m, NAN) for r in rs) for m in ('followShare', 'soloSolves')})
            S['mechanism'][(v, ch, N, name)] = mech
    return S


def writeTrackA(path, S, meta):
    "report.md for Track A: provenance first, then the TRACKA_PLAN tables (exploratory)."
    L = [f"# Phase 12, Track A: instructions and channels for Claude ({meta['mode']})", '',
         'Exploratory, pre-specified in `TRACKA_PLAN`.', '',
         f"- Plan hash `{meta['planHash']}`, git commit `{meta['commit']}`; model {meta['model']}; prices checked "
         f"{meta['pricesChecked']}; cost ${meta['cost']:.2f}; total API spend ${meta['spent']:.2f} of the ${meta['cap']:.0f} "
         f"cap; calls {meta['calls']}; wall time {meta['wall']:.0f} s", '',
         '## Effect of each variant (minus the Phase 11 cell, paired on seed)', '',
         '| variant | channel | N | welfare difference | coordinator minus Phase 11 blackboard |', '|---|---|---|---|---|']
    L += [f"| {v} | {ch} | {N} | {_fmt(e)} | {_fmt(S['gap'][(v, ch, N)]) if (v, ch, N) in S['gap'] else '-'} |"
          for (v, ch, N), e in S['effect'].items()]
    L += ['', '## Mechanism (variant vs Phase 11, means)', '',
          '| variant | channel | N | run | welfare | shares per held clue | redundant shares | follow share | solo solves |',
          '|---|---|---|---|---|---|---|---|---|']
    g = lambda d, m: f"{d[m]['mean']:.2f}" if m in d else '-'
    L += [f"| {v} | {ch} | {N} | {name} | {g(d, 'welfare')} | {g(d, 'sharePerHeld')} | {g(d, 'redundantShares')} | "
          f"{g(d, 'followShare')} | {g(d, 'soloSolves')} |" for (v, ch, N, name), d in S['mechanism'].items()]
    path.write_text('\n'.join(L) + '\n')
    return path


def figTrackA(out, S):
    "coopA.png: each variant's welfare difference from the Phase 11 cell, with 95% intervals."
    items = list(S['effect'].items())
    fig, ax = plt.subplots(figsize=(8, 0.4 * len(items) + 1.2))
    m = np.array([v['mean'] for _, v in items])
    err = [m - np.array([v['ci'][0] for _, v in items]), np.array([v['ci'][1] for _, v in items]) - m]
    ax.errorbar(m, range(len(items)), xerr=err, fmt='o', capsize=3)
    ax.set_yticks(range(len(items)), [f'{v} {ch} N={N}' for (v, ch, N), _ in items], fontsize=8)
    ax.axvline(0, color='k', lw=0.5)
    ax.invert_yaxis()
    ax.set(xlabel='welfare per agent-turn minus Phase 11', title='Track A: instruction and channel variants')
    return _save(fig, out / 'coopA.png')

## Tests

In [ ]:
import math, tempfile
from pathlib import Path
from explRL.bench_grid import rowKey

# synthetic rows: variant cells sit exactly 0.1 above the Phase 11 cells
def row(study, level, ch, N, seed, w):
    return dict(study=study, factor=study, level=level, channel=ch, N=N, seed=seed, budget='turns', episode=0, runner='llm',
                welfare=w, sharePerHeld=0.5, redundantShares=1.0)
p11 = [row('clue', 'clue', ch, N, s, 0.3 + 0.01 * s % 7 + (0.2 if ch == 'blackboard' else 0)) for ch in ('blackboard', 'hierarchical')
       for N in (4, 8) for s in range(900, 905)]
a = [row('coopA', 'A1', 'hierarchical', N, s, 0.3 + 0.01 * s % 7 + 0.1) for N in (4, 8) for s in range(900, 905)]
logm = {rowKey(r): dict(followShare=0.8, soloSolves=1) for r in a}
S = trackASummaries(a, p11, logm, {})
assert abs(S['effect'][('A1', 'hierarchical', 4)]['mean'] - 0.1) < 1e-12 and abs(S['gap'][('A1', 'hierarchical', 4)]['mean'] + 0.1) < 1e-12
assert S['mechanism'][('A1', 'hierarchical', 4, 'variant')]['followShare']['mean'] == 0.8
assert math.isnan(S['mechanism'][('A1', 'hierarchical', 4, 'Phase 11')]['followShare']['mean'])
with tempfile.TemporaryDirectory() as d:
    meta = dict(mode='quick', planHash='abc', commit='x', model='claude-haiku-5-5', pricesChecked='2026-10-12', cost=0.0,
                spent=44.29, cap=150, calls=0, wall=1.0)
    text = writeTrackA(Path(d) / 'report.md', S, meta).read_text()
    assert text.index('abc') < text.index('## Effect') and '| A1 | hierarchical | 4 |' in text
    assert figTrackA(Path(d), S).stat().st_size > 0

# logMetrics agrees with the board's own solo-solve count and with coopMetrics' follow share on real rollouts
import json
from explRL.bench_grid import rowKey as rk
from explRL.clue_policy import claudePolicy
from explRL.clue_roll import coopRows, rollout
from explRL.llm_client import LLMClient
from explRL.llm_fake import FakeClient, fakeReply
from explRL.llm_runs import LLMConfig
import re, zlib
from explRL.clue_policy import scriptedPolicy
def guesser(system, blocks):
    "Opens the first listed subtask, then submits one of the candidates its known clues leave (trial and error)."
    o = blocks[-1]['text']
    m = re.search(r'Your open subtask: T(\d+), open\. Find x with 0 <= x < (\d+)\. Clues you know: (.*)\.', o)
    if not m:
        return json.dumps(dict(action='open', target=re.search(r'\): (T\d+)', o).group(1), answer=0, clue='', notes=''))
    cl = [tuple(map(int, c)) for c in re.findall(r'x mod (\d+) = (\d+)', m.group(3))]
    xs = [y for y in range(int(m.group(2))) if all(y % p == r for p, r in cl)]
    return json.dumps(dict(action='submit', target='', answer=xs[zlib.crc32(o.encode()) % len(xs)], clue='', notes=''))
fakePol = lambda: claudePolicy(LLMClient('claude-haiku-5-5', client=FakeClient(fakeReply)))
solos = 0
for ch, pol in (('hierarchical', fakePol()), ('blackboard', fakePol()), ('blackboard', scriptedPolicy(guesser))):
    cfg = LLMConfig(task='clue', channel=ch, N=4, turns=25, world=dict(m=2, r=1, R=30))
    ro = rollout(cfg, [900, 901], pol, memory='full')
    rows = coopRows(cfg, ro)
    recs = [dict(key=f'k{b}', **x) for b in range(2) for lg in ro.logs[b] for x in lg]
    lm = logMetrics(recs, 2)
    for b in range(2):
        assert lm[f'k{b}']['soloSolves'] == rows[b]['soloSolves']
        f1, f2 = lm[f'k{b}']['followShare'], rows[b]['followShare']
        assert (math.isnan(f1) and math.isnan(f2)) or f1 == f2
        solos += rows[b]['soloSolves']
assert solos > 0                                    # the guesser makes solo solves, so the agreement is not vacuous

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()